### Connect to MongoDB

In [ ]:
import requests
import time
import pandas as pd

AQS_USER  = userdata.get('AQS_USER')
AQS_KEY   = userdata.get('AQS_KEY')
MONGO_URI = userdata.get('MONGO_URI2')

LOCATIONS = [
    {"state": "06", "county": "037", "city": "Los Angeles"},
    {"state": "36", "county": "061", "city": "New York"},
    {"state": "48", "county": "201", "city": "Houston"},
    {"state": "17", "county": "031", "city": "Chicago"},
    {"state": "04", "county": "013", "city": "Phoenix"},
]

BDATE = "20220101"
EDATE = "20231231"
PARAM = "88101"  # PM2.5

### Data Creation

In [ ]:
def aqi_category(aqi_val):
    if aqi_val is None: return "Unknown"
    if aqi_val <= 50: return "Good"
    if aqi_val <= 100: return "Moderate"
    if aqi_val <= 150: return "Unhealthy for Sensitive Groups"
    if aqi_val <= 200: return "Unhealthy"
    if aqi_val <= 300: return "Very Unhealthy"
    return "Hazardous"

client     = MongoClient(MONGO_URI)
collection = client["air_quality_db"]["daily_aqi"]

YEARS = [("20220101", "20221231"), ("20230101", "20231231")]

for loc in LOCATIONS:
    for bdate, edate in YEARS:
        resp = requests.get("https://aqs.epa.gov/data/api/dailyData/byCounty", params={
            "email": AQS_USER,
            "key": AQS_KEY,
            "param": PARAM,
            "bdate": bdate,
            "edate": edate,
            "state": loc["state"],
            "county": loc["county"],
        })

        data = resp.json()
        if data["Header"][0]["status"] == "Failed":
            print(f"FAILED {loc['city']} {bdate[:4]}: {data['Header'][0]['error']}")
            continue

        records = data["Data"]
        docs = [{
            "city": loc["city"],
            "date": r["date_local"],
            "location": {
                "lat": r["latitude"],
                "lon": r["longitude"],
                "site": r["local_site_name"],
                "address": r["site_address"],
            },
            "pollutant": {
                "parameter": r["parameter"],
                "units": r["units_of_measure"],
                "mean": r["arithmetic_mean"],
                "max_value": r["first_max_value"],
                "max_hour": r["first_max_hour"],
            },
            "aqi": {
                "value": r["aqi"],
                "category": aqi_category(r["aqi"]),
            },
            "observation_count": r["observation_count"],
            "observation_percent": r["observation_percent"],
            "source": "EPA AQS API",
        } for r in records]

        collection.insert_many(docs)
        print(f"{loc['city']} {bdate[:4]}: inserted {len(docs)} documents")
        time.sleep(5)